# 🚀 AI톡 코치 - QLoRA Fine-tuning

한국어 커플 상담 특화 모델 학습

**Runtime**: T4 GPU (16GB VRAM) 이상 필요
**소요 시간**: 약 30~50분
**모델**: Llama 3.2 3B Instruct

## 1. 환경 설정

In [ ]:
# GPU 확인
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# HuggingFace 로그인 (토큰 필요)
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')
# !huggingface-cli login --token $HF_TOKEN

In [ ]:
# 필수 라이브러리 설치
!pip install -q transformers peft bitsandbytes accelerate datasets scipy
!pip install -q trl>=0.4.0

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
import torch

print("✅ Libraries installed")

## 2. 데이터셋 업로드

In [ ]:
# 방법 1: 파일 직접 업로드
from google.colab import files
uploaded = files.upload()  # korean_counseling_dataset.json 선택

# 방법 2: Git에서 클론
# !git clone https://github.com/YOUR_USERNAME/cash1.git
# data_path = "./cash1/ai-engine/training/korean_counseling_dataset.json"

import json
with open(list(uploaded.keys())[0], 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ Dataset loaded: {len(data)} examples")

## 3. 프롬프트 템플릿 정의

In [ ]:
def format_instruction(example):
    """QLoRA 학습용 프롬프트 포맷"""
    category = example.get("analysis_type", "comprehensive")
    
    system_prompts = {
        "emotion": "당신은 20년 경력의 커플 상담사입니다. 다음 대화의 감정/분위기를 분석해주세요. 한국인의 감정 표현과 문화적 맥락을 깊이 이해하고 있습니다.",
        "interest": "당신은 20년 경력의 커플 상담사입니다. 상대방의 관심도를 분석해주세요. 한국인의 연락 패턴과 관심 표현 방식을 이해하고 있습니다.",
        "replies": "당신은 20년 경력의 커플 상담사입니다. 상황에 맞는 자연스러운 한국어 답장을 추천해주세요. 반말/존댓말 구분과 데이트 문화 맥락을 고려합니다.",
        "advice": "당신은 20년 경력의 커플 상담사입니다. 대화 방식의 개선점을 조언해주세요. 한국인의 소통 스타일과 문화적 특성을 반영합니다.",
        "comprehensive": "당신은 20년 경력의 커플 상담사입니다. 대화를 종합적으로 분석해주세요. 감정, 관심도, 조언, 답장 추천을 포함합니다.",
    }
    
    system = system_prompts.get(category, system_prompts["comprehensive"])
    conversation = example["conversation"].strip()
    response_json = json.dumps(example["response"], ensure_ascii=False)
    explanation = example.get("explanation", "")
    
    # Llama 3 chat template
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system}

응답은 반드시 JSON 형식으로 작성해주세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

[대화]
{conversation}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response_json}

[분석 근거]
{explanation}<|eot_id|>"""
    
    return {"text": text}

# 데이터셋 포맷
dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction)
print(f"Dataset: {len(dataset)} examples")

## 4. QLoRA 모델 로드

In [ ]:
# 모델 설정
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit 양자화 설정 (T4에서 돌아가도록 최적화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model with 4-bit quantization...")

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "14GiB", "cpu": "30GiB"},
    trust_remote_code=True,
)

# QLoRA 준비
model = prepare_model_for_kbit_training(model)

print(f"Model loaded! GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. LoRA 설정 적용

In [ ]:
# LoRA 설정 (QLoRA에 최적화된 설정)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,                    #较大的 r = 더 많은 fine-tune 파라미터
    lora_alpha=128,          # r * 2
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head",
    ],
    bias="none",
)

# LoRA 적용
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 메모리 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 6. 토큰화

In [ ]:
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

dataset = dataset.map(tokenize, batched=False, remove_columns=["text"])
print(f"Dataset tokenized: {len(dataset)} examples")

## 7. TrainingArguments 설정

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qlora_output",
    num_train_epochs=15,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,   # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=0.5,
    logging_steps=5,
    save_steps=50,
    eval_steps=50,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    report_to="none",
    seed=42,
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

## 8. 학습 시작 🚀

In [ ]:
print("Starting QLoRA training...")
print(f"Dataset size: {len(dataset)}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total steps: {len(dataset) * 15 / 16}")

trainer.train()

print("✅ Training complete!")

## 9. 모델 저장

In [ ]:
# LoRA 어댑터 저장
trainer.save_model("./qlora_output/final")
tokenizer.save_pretrained("./qlora_output/final")

# Merged 모델 저장 (전체 모델)
print("Merging LoRA weights...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./qlora_output/merged")
tokenizer.save_pretrained("./qlora_output/merged")

print("✅ Model saved to ./qlora_output/")

## 10. 모델 다운로드

In [ ]:
# merged 모델을 zip으로 압축
!cd ./qlora_output/merged && zip -r ../../../../aitalkcoach_qlora_model.zip .

# 다운로드
from google.colab import files
files.download("./aitalkcoach_qlora_model.zip")

print("Model downloaded!")

---

## 📁 출력 파일

- `qlora_output/final/` - LoRA 어댑터 (가벼움, ~50MB)
- `qlora_output/merged/` - 전체 모델 (무거움, ~6GB)
- `aitalkcoach_qlora_model.zip` - 다운로드용 압축파일

## 📦 다음 단계

1. 모델을 `ai-engine/models/`에 배치
2. `on_device_model.js`로 연동
3. APK 빌드하여手机上 테스트